In [0]:
%pip install azure-eventhub

In [0]:
dbutils.library.restartPython()

In [0]:
import time
import json
import random
from azure.eventhub import EventHubProducerClient, EventData

CONNECTION_STR = "Endpoint=sb://batteryhealtheventhub.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=Ht+Mnt/MJYZXQSkTSKV6yCQuNAQGpNJsh+AEhMus5ik="
EVENTHUB_NAME = "realtime-sensor-data"

producer = EventHubProducerClient.from_connection_string(CONNECTION_STR, eventhub_name=EVENTHUB_NAME)

print("Starting Producer... (Press Stop to end)")

try:
    with producer:
        while True:
            # Generate data
            v = round(random.uniform(3.0, 4.5), 2) if random.random() > 0.1 else "ERR_VOLT"
            data = {
                "battery_id": "BATT_01",
                "voltage": v,
                "temp": round(random.uniform(20, 80), 1),
                "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
            }
            
            # Send data
            event_batch = producer.create_batch()
            event_batch.add(EventData(json.dumps(data)))
            producer.send_batch(event_batch)
            
            print(f"Sent: {data}")
            
            # SAVE CREDITS: Wait 10 seconds instead of 1
            time.sleep(10) 
except KeyboardInterrupt:
    print("Producer stopped.")